# Inicializando os dados csv

In [1]:
dados = '''
Idade,Sexo,Pressão Sanguínea,Colesterol,Batimentos Cardíacos,Nível de Açúcar no Sangue,Histórico Familiar,Ataque Cardíaco
45,M,130,200,80,100,Sim,Sim
55,F,145,240,70,150,Não,Sim
60,M,120,180,90,90,Sim,Não
50,M,135,220,75,160,Sim,Sim
40,F,125,190,85,95,Não,Não
'''

O `pd.read_csv()` precisa de um caminho de arquivo ou área de memória para ler
e o `StringIO()` permite que um string seja tratada como área de memória (buffer)

In [2]:
import pandas as pd
from io import StringIO

df = pd.read_csv(StringIO(dados))

In [3]:
df = pd.read_csv('https://gist.githubusercontent.com/caiohamamura/d3ac8997a575cfed876753bdd750a6fb/raw/57407983cf9c23e7b0712d81b9bd6e446e880842/cardiaco.csv')

In [4]:
df

,Idade,Sexo,Pressão Sanguínea,Colesterol,Batimentos Cardíacos,Nível de Açúcar no Sangue,Histórico Familiar,Ataque Cardíaco
0,45,M,130,200,80,100,Sim,Sim
1,55,F,145,240,70,150,Não,Sim
2,60,M,120,180,90,90,Sim,Não
3,50,M,135,220,75,160,Sim,Sim
4,40,F,125,190,85,95,Não,Não


# Recodificar para 0 e 1

Codificar os campos categóricos em números

In [5]:
df['Sexo'] = df['Sexo'].map({'M': 1, 'F': 0})
df['Ataque Cardíaco'] = df['Ataque Cardíaco'].map({'Sim': 1, 'Não': 0})
df['Histórico Familiar'] = df['Histórico Familiar'].map({'Sim': 1, 'Não': 0})

In [6]:
df

,Idade,Sexo,Pressão Sanguínea,Colesterol,Batimentos Cardíacos,Nível de Açúcar no Sangue,Histórico Familiar,Ataque Cardíaco
0,45,1,130,200,80,100,1,1
1,55,0,145,240,70,150,0,1
2,60,1,120,180,90,90,1,0
3,50,1,135,220,75,160,1,1
4,40,0,125,190,85,95,0,0


# Alterar as escalas das colunas para 0 a 1

In [7]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
colunas_numericas = ['Idade', 'Pressão Sanguínea', 'Colesterol', 'Batimentos Cardíacos', 'Nível de Açúcar no Sangue']
df[colunas_numericas] = scaler.fit_transform(df[colunas_numericas])

In [8]:
df

,Idade,Sexo,Pressão Sanguínea,Colesterol,Batimentos Cardíacos,Nível de Açúcar no Sangue,Histórico Familiar,Ataque Cardíaco
0,0.25,1,0.4,0.333333,0.50,0.142857,1,1
1,0.75,0,1.0,1.000000,0.00,0.857143,0,1
2,1.00,1,0.0,0.000000,1.00,0.000000,1,0
3,0.50,1,0.6,0.666667,0.25,1.000000,1,1
4,0.00,0,0.2,0.166667,0.75,0.071429,0,0


In [12]:
import torch
import torch.optim.optimizer

# Transforma os dados de entrada e saída para tensores do PyTorch
entrada_tensor = torch.tensor(df.drop('Ataque Cardíaco', axis=1).values, dtype=torch.float)
saida_tensor = torch.tensor(df['Ataque Cardíaco'].values, dtype=torch.float).view(-1, 1)


# Inicializa pesos aleatórios
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

weights = torch.randn(entrada_tensor.shape[1], 1, dtype=torch.float, requires_grad=True)
bias = torch.randn(1, dtype=torch.float, requires_grad=True)

print (f'Pesos iniciais: {weights}')
print (f'Bias inicial: {bias}')

Pesos iniciais: tensor([[ 0.3367],
        [ 0.1288],
        [ 0.2345],
        [ 0.2303],
        [-1.1229],
        [-0.1863],
        [ 2.2082]], requires_grad=True)
Bias inicial: tensor([-0.6380], requires_grad=True)


In [ ]:
# Taxa de aprendizagem
learning_rate = 1
num_epochs = 1000

# Função de perda
loss_fn = torch.nn.BCELoss()

# Otimizador (Stochastic Gradient Descent)
optimizer = torch.optim.SGD([weights, bias], lr=learning_rate)

melhores_pesos = None
melhor_loss = float('inf')

for epoch in range(num_epochs):
    optimizer.zero_grad()
    y_pred = torch.sigmoid(entrada_tensor @ weights + bias)
    loss = loss_fn(y_pred, saida_tensor)



    loss.backward()
    optimizer.step()
    #print('Updates in weigths:' + str(weights.grad))
    #print('Updates in bias:' + str(bias.grad))
    #print('New weights:' + str(weights))

    if (epoch) % 100 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

print (f'Initial weights: {weights} \nInitial bias: {bias}')

Epoch [1/1000], Loss: 0.5198
Epoch [101/1000], Loss: 0.0472
Epoch [201/1000], Loss: 0.0256
Epoch [301/1000], Loss: 0.0175
Epoch [401/1000], Loss: 0.0133
Epoch [501/1000], Loss: 0.0107
Epoch [601/1000], Loss: 0.0089
Epoch [701/1000], Loss: 0.0077
Epoch [801/1000], Loss: 0.0067
Epoch [901/1000], Loss: 0.0060
Initial weights: tensor([[-2.9370],
        [ 2.3137],
        [ 4.5061],
        [ 4.1350],
        [-7.0232],
        [ 2.5391],
        [ 4.3930]], requires_grad=True) 
Initial bias: tensor([-1.5722], requires_grad=True)


In [ ]:
weights

tensor([[-2.9370],
        [ 2.3137],
        [ 4.5061],
        [ 4.1350],
        [-7.0232],
        [ 2.5391],
        [ 4.3930]], requires_grad=True)

In [ ]:
bias

tensor([-1.5722], requires_grad=True)

In [ ]:
torch.sigmoid(entrada_tensor @ weights + bias)

tensor([[0.9883],
        [0.9991],
        [0.0080],
        [1.0000],
        [0.0063]], grad_fn=<SigmoidBackward0>)

In [ ]:
entrada_tensor

tensor([[0.2500, 1.0000, 0.4000, 0.3333, 0.5000, 0.1429, 1.0000],
        [0.7500, 0.0000, 1.0000, 1.0000, 0.0000, 0.8571, 0.0000],
        [1.0000, 1.0000, 0.0000, 0.0000, 1.0000, 0.0000, 1.0000],
        [0.5000, 1.0000, 0.6000, 0.6667, 0.2500, 1.0000, 1.0000],
        [0.0000, 0.0000, 0.2000, 0.1667, 0.7500, 0.0714, 0.0000]])